In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import json
import re
import math
from glob import glob
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVR, SVC
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score, confusion_matrix, classification_report, accuracy_score, f1_score
from xgboost import XGBRegressor, XGBClassifier, cv, DMatrix, plot_importance
import statsmodels.api as sm
import seaborn as sns
from copy import deepcopy
import joblib

## **Load District-AgroClimatic Zone Mapping**

In [3]:
# import ee

# ee.Authenticate()
# ee.Initialize(project='ee-mtpictd')

# agro_gee = ee.FeatureCollection("projects/ee-mtpictd/assets/harsh/Agroclimatic_regions")
# agro_df = ee.data.computeFeatures({
#     'expression': agro_gee,
#     'fileFormat': 'GEOPANDAS_GEODATAFRAME'
# })
# agro_df.to_csv("drive/MyDrive/dhruvi/data/agroclimatic_zone.csv", index=False)

In [4]:
agro_df = gpd.read_file("drive/MyDrive/dhruvi/data/agroclimatic_zone.csv", GEOM_POSSIBLE_NAMES="geometry", KEEP_GEOM_COLUMNS="NO")

In [5]:
set(agro_df['regionname'])

{'Central Plateau & Hills Region',
 'East Coast Plains & Hills Region',
 'Eastern Himalayan Region',
 'Eastern Plateau & Hills Region',
 'Gujarat Plains & Hills Region',
 'Island region',
 'Lower Gangetic Plain Region',
 'Middle Gangetic Plain Region',
 'Southern Plateau and Hills Region',
 'Trans Gangetic Plain Region',
 'Upper Gangetic Plain Region',
 'West Coast Plains & Ghat Region',
 'Western Dry Region',
 'Western Himalayan Region',
 'Western Plateau and Hills Region'}

In [6]:
# agro_dist_df = gpd.read_file("drive/MyDrive/dhruvi/data/distict_agroclimate_map.csv", GEOM_POSSIBLE_NAMES="geometry", KEEP_GEOM_COLUMNS="NO")
agro_dist_df = gpd.read_file("drive/MyDrive/harsh/district_to_agroclimaticZone_mapping.csv", GEOM_POSSIBLE_NAMES="geometry", KEEP_GEOM_COLUMNS="NO")
agro_dist_df.rename(columns={'District': 'Name', 'IntersectingZones': 'intersecting_zones', 'IntersectingAreas': 'common_area', 'AgroclimaticZone': 'agroclimatic_zone'}, inplace = True)

In [7]:
agro_dist_df['agroclimatic_zone'].unique()

array(['None', 'Southern Plateau and Hills Region',
       'East Coast Plains & Hills Region', 'Eastern Himalayan Region',
       'Middle Gangetic Plain Region', 'Trans Gangetic Plain Region',
       'Eastern Plateau & Hills Region', 'Gujarat Plains & Hills Region',
       'West Coast Plains & Ghat Region', 'Western Himalayan Region',
       'Western Plateau and Hills Region',
       'Central Plateau & Hills Region', 'Western Dry Region',
       'Upper Gangetic Plain Region', 'Lower Gangetic Plain Region'],
      dtype=object)

## **Common Utilities**

In [8]:
def get_model_paths(GEDI_DATA):
    if GEDI_DATA == 'CHM':
        # MODEL_PATH = "drive/MyDrive/dhruvi/best_models/corrected/"
        MODEL_PATH = ""
        dist_data_path = "drive/MyDrive/gedi_chm_season_data_2020/"
        dist_data_path_19 = "drive/MyDrive/gedi_chm_season_data_2019/"
        dist_data_path_21 = "drive/MyDrive/gedi_chm_season_data_2021/"
        # gedi_features = ['rh10', 'rh25', 'rh50', 'rh75', 'rh90', 'rh95', 'rh98', 'rh100', 'delta_time']
        gedi_features = ['rh50', 'rh75', 'rh98', 'delta_time']
    else:
        # MODEL_PATH = "drive/MyDrive/dhruvi/best_models/corrected/"
        MODEL_PATH = "drive/MyDrive/Anoushka/canopy_density/best_models/"
        dist_data_path = "drive/MyDrive/gedi_cc_season_data_2020/"
        dist_data_path_19 = "drive/MyDrive/gedi_cc_season_data_2019/"
        dist_data_path_21 = "drive/MyDrive/gedi_cc_season_data_2021/"
        gedi_features = ['cover', 'delta_time']
    return MODEL_PATH, dist_data_path, dist_data_path_19, dist_data_path_21, gedi_features

In [9]:
base_features = ['VV', 'VH', 'angle', 'VV_VH_Ratio', 'VH_VV_Ratio', 'SAR_NDVI', 'SAR_DVI', 'SAR_SVI', 'SAR_RDVI', 'SAR_NRDVI', 'SAR_SSDVI',
                'B1','B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B11','B12', 'NDVI', 'NDWI', 'EVI', 'OSAVI', 'ARVI', 'VARI']

seasons = ['kharif', 'rabi', 'zaid']

features = [feature + '_' + season for season in seasons for feature in base_features] + ['month_sin', 'month_cos']

In [10]:
def get_dist_data(dist_name, data_path, file_suffix, files):
  df = None
  for file in dist_name:
    dist_file = data_path + file + file_suffix
    if dist_file in files:
      try:
        data = pd.read_csv(dist_file)
        if df is None:
          df = data
        else:
          df = pd.concat([df, data], ignore_index=True)
      except Exception as exp:
        # print("Error reading file ", dist_file, " - ", exp)
        continue
    # else:
    #   print("No data found for ", dist_file)
  return df

In [11]:
def add_s1_indices(df):
    df['VV_VH_Ratio'] = df['VV'] / df['VH']
    df['VH_VV_Ratio'] = df['VH'] / df['VV'] # SAR Ratio Vegetation Index
    df['SAR_NDVI'] = (df['VH'] - df['VV']) / (df['VH'] + df['VV'])
    df['SAR_DVI'] = df['VH'] - df['VV']
    df['SAR_SVI'] = df['VH'] + df['VV']
    df['SAR_RDVI'] = (df['VH'] / df['VV']) - (df['VV'] / df['VH'])
    df['SAR_NRDVI'] = (df['VH']/df['VV'] - df['VV']/df['VH']) / (df['VH']/df['VV'] + df['VV']/df['VH'])
    df['SAR_SSDVI'] = (df['VH'])**2 - (df['VV'])**2
    # df['SRVI_VV'] = df.apply(lambda x: (x['VV'])**2 / sqrt(x['VH']), axis=1)
    # df['SRVI_VH'] = df.apply(lambda x: (x['VH'])**2 / sqrt(x['VV']), axis=1)

def add_s2_indices(df):
    df['NDVI'] = (df['B8'] - df['B4']) / (df['B8'] + df['B4'])
    df['NDWI'] = (df['B8'] - df['B12']) / (df['B8'] + df['B12'])
    df['EVI'] = (2.5 * ((df['B8'] - df['B4']) / (df['B8'] + 6*df['B4'] - 7.5*df['B2'] + 1)))
    df['OSAVI'] = (df['B8'] - df['B4']) / (df['B8'] + df['B4'] + 0.16)
    df['ARVI'] = (df['B8'] - 2*df['B4'] + df['B2']) / (df['B8'] + 2*df['B4'] + df['B2'])
    df['VARI'] = (df['B3'] - df['B4']) / (df['B3'] + df['B4'] - df['B2'])
    # df['VARI'] = df.apply(lambda x: (x['B3'] - x['B4']) / (x['B3'] + x['B4'] - x['B2']) if x['B3']+x['B4']-x['B2']!=0 else np.nan, axis=1)

def encode(data, col, max_val):
    data[col + '_sin'] = np.sin(2 * np.pi * data[col]/max_val)
    data[col + '_cos'] = np.cos(2 * np.pi * data[col]/max_val)
    return data

In [12]:
def get_rs_data(region_name, dist_data_path, files, gedi_features):
  dist_name = list(agro_dist_df[agro_dist_df['intersecting_zones'].str.contains(region_name, regex=False)]['Name'].apply(lambda x: re.sub("[^A-Za-z0-9]","",x)))
  print("Total Districts: ", len(dist_name))

  # season_df = dict()
  df = None
  for season in seasons:
    s1_df = get_dist_data(dist_name, dist_data_path, "_s1_"+season+".csv", files).drop(columns=['system:index']).drop_duplicates('.geo')
    print("Total Sentinel-1 Data of ", season, ": ", len(s1_df))
    s2_df = get_dist_data(dist_name, dist_data_path, "_s2_"+season+".csv", files).drop(columns=['system:index']).drop_duplicates('.geo')
    print("Total Sentinel-2 Data of ", season, ": ", len(s2_df))
    season_df = s1_df.merge(s2_df, on='.geo').dropna()
    # if len(season_df) > 150_000:
    #   season_df = season_df.sample(n=150_000, random_state=42, ignore_index=True)
    add_s1_indices(season_df)
    add_s2_indices(season_df)
    season_df.rename(columns={band:band+"_"+season for band in base_features}, inplace=True)
    if df is None:
      df = season_df
    else:
      season_df.drop(columns=[f for f in gedi_features if f in season_df.columns], inplace=True)
      df = df.merge(season_df, on='.geo')

  df.replace([np.inf, -np.inf], np.nan, inplace=True)
  df.dropna(inplace=True)

  df['month'] = df['delta_time'].apply(lambda x: int((x / 86400 / 30) % 12 + 1))
  df = encode(df, 'month', 12)
  df[['lon', 'lat']] = pd.DataFrame(df['.geo'].apply(lambda x: json.loads(x)['coordinates']).tolist(), index=df.index)
  df.drop(columns=['.geo', 'delta_time'], inplace=True)
  print("Merged Data Size: ", len(df))

  df['geometry'] = df.apply(lambda x: Point(x['lon'], x['lat']), axis=1)
  gdf = gpd.GeoDataFrame(df)
  return gdf[gdf.within(agro_df[agro_df['regionname'] == region_name].iloc[0].geometry)]


def clean_data(df, y_col):
  # print("Mean: ", df[y_col].mean())
  # print("Std: ", df[y_col].std())
  # print("Range: ", df[y_col].mean() - 3*df[y_col].std(), df[y_col].mean() + 3*df[y_col].std())

  # Remove outliers beyond 3 std deviations
  clean_df = df[(df[y_col] >= df[y_col].mean() - 3*df[y_col].std()) &
                (df[y_col] <= df[y_col].mean() + 3*df[y_col].std())].reset_index(drop=True)
  return clean_df


def bin_sample(cdf, default_samples, y_col):
  bins = 6
  cdf['bin'] = None
  # mean_ = cdf[y_col].mean()
  # std_ = cdf[y_col].std()
  min_ = math.floor(cdf[y_col].min())
  max_ = math.ceil(cdf[y_col].max())
  std_ = math.floor((max_ - min_) / bins)
  print(min_, max_, std_)
  # print(mean_, std_)

  i = 1
  for x in range(1, bins+1):
  # for x in range(max(math.floor(mean_ - 3*std_),0), math.ceil(mean_ + 3*std_), math.ceil(std_)):
    lb = min_ + (i-1)*std_
    ub = min_ + i*std_
    # lb = x
    # ub = x + math.ceil(std_)
    if i==1:
      cdf.loc[(cdf[y_col] < ub), 'bin'] = i
    elif i==6:
      cdf.loc[(cdf[y_col] >= lb), 'bin'] = i
    else:
      cdf.loc[(cdf[y_col] >= lb) & (cdf[y_col] < ub), 'bin'] = i
    samples = len(cdf[cdf['bin'] == i])
    print(lb, ub, samples)
    i += 1

  min_samples = min(cdf['bin'].value_counts())

  samples = max(default_samples, min_samples)
  sample_df = None
  for i in range(1, bins+1):
    bin_df = cdf[cdf['bin'] == i]
    replace = False
    if samples > len(bin_df):
      replace = True
    print(i, len(bin_df), samples, replace)
    if sample_df is None:
      sample_df = bin_df.sample(n=samples, random_state=42, ignore_index=True, replace=replace)
    else:
      sample_df = pd.concat([sample_df, bin_df.sample(n=samples, random_state=42, ignore_index=True, replace=replace)], ignore_index=True)
      # if i in [bins/2, bins/2+1]:
      #   sample_df = pd.concat([sample_df, bin_df], ignore_index=True)
      # else:
      #   sample_df = pd.concat([sample_df, bin_df.sample(n=samples, random_state=42, ignore_index=True, replace=replace)], ignore_index=True)

  print("Dataset Size after bin-based sampling: ", len(sample_df))

  return sample_df

In [13]:
# Classification specific functions

def get_class(data_value, threshold):
  if data_value <= threshold:
    return 0
  else:
    return 1


def balance_class(season_df, y_col):
  min_samples = min(season_df[y_col].value_counts())
  print(min_samples)

  sample_df = None
  for c in season_df[y_col].unique():
    print(c)
    if sample_df is None:
      sample_df = season_df[season_df[y_col] == c].sample(n=min_samples, random_state=42, ignore_index=True, replace=False)
    else:
      sample_df = pd.concat([sample_df, season_df[season_df[y_col] == c].sample(n=min_samples, random_state=42, ignore_index=True, replace=False)], ignore_index=True)

  return sample_df


def xgb_cv(dtrain, dtest, predictors, y_col, useTrainCV=True, cv_folds=5, early_stopping_rounds=20):
    alg = XGBClassifier(
        learning_rate=0.1,
        n_estimators=1000,
        max_depth=5,
        min_child_weight=1,
        gamma=0,
        subsample=0.8,
        colsample_bytree=0.8,
        objective= 'binary:logistic',
        nthread=4,
        scale_pos_weight=1,
        seed=27,
        eval_metric='auc'
    )

    if useTrainCV:
        xgb_param = alg.get_xgb_params()
        xgtrain = DMatrix(dtrain[predictors].values, label=dtrain[y_col].values)
        cvresult = cv(xgb_param, xgtrain, num_boost_round=alg.get_params()['n_estimators'], nfold=cv_folds,
            metrics='auc', early_stopping_rounds=early_stopping_rounds)
        alg.set_params(n_estimators=cvresult.shape[0])

    #Fit the algorithm on the data
    alg.fit(dtrain[predictors], dtrain[y_col]
            # ,eval_metric='auc'
            )

    #Predict training set:
    pred_y_train = alg.predict(dtrain[predictors])
    pred_y_train_prob = alg.predict_proba(dtrain[predictors])[:,1]

    #Print model report:
    print("**Train Set**")
    print(confusion_matrix(dtrain[y_col], pred_y_train))
    print(classification_report(dtrain[y_col], pred_y_train))
    print("Accuracy :", accuracy_score(dtrain[y_col].values, pred_y_train)*100)
    print("F1 Score :", f1_score(dtrain[y_col].values, pred_y_train, average='macro')*100)

    if dtest is None:
        pred_y_test = []
    else:
        pred_y_test = alg.predict(dtest[predictors])
        print("**Test Set**")
        print(confusion_matrix(dtest[y_col], pred_y_test))
        print(classification_report(dtest[y_col], pred_y_test))
        print("Accuracy :", accuracy_score(dtest[y_col].values, pred_y_test)*100)
        print("F1-Score :", f1_score(dtest[y_col].values, pred_y_test, average='macro')*100)

    return alg, pred_y_train, pred_y_test

In [14]:
def plot_month_performance(clean_df, clean_df21, xgb1, features, y_col):
    month_acc = []
    for month in range(1,13):
        month_19 = clean_df[(clean_df['year'] == 2019) & (clean_df['month'] == month)]
        month_20 = clean_df[(clean_df['year'] == 2020) & (clean_df['month'] == month)]
        month_21 = clean_df21[clean_df21['month'] == month]
        pred_y_19 = xgb1.predict(month_19[features])
        pred_y_20 = xgb1.predict(month_20[features])
        pred_y_21 = xgb1.predict(month_21[features])
        month_acc.append({
            'month': month,
            '2019_samples': len(month_19),
            '2020_samples': len(month_20),
            '2021_samples': len(month_21),
            '2019_acc': round(accuracy_score(month_19[y_col], pred_y_19)*100 if len(pred_y_19) > 0 else 0, 0),
            '2020_acc': round(accuracy_score(month_20[y_col], pred_y_20)*100 if len(pred_y_20) > 0 else 0, 0),
            '2021_acc': round(accuracy_score(month_21[y_col], pred_y_21)*100 if len(pred_y_21) > 0 else 0, 0),
            '2019_f1': round(f1_score(month_19[y_col], pred_y_19)*100 if len(pred_y_19) > 0 else 0, 0),
            '2020_f1': round(f1_score(month_20[y_col], pred_y_20)*100 if len(pred_y_20) > 0 else 0, 0),
            '2021_f1': round(f1_score(month_21[y_col], pred_y_21)*100 if len(pred_y_21) > 0 else 0, 0),
            })
    month_df = pd.DataFrame(month_acc)
    print(month_df['2019_acc'].mean(), month_df['2020_acc'].mean(), month_df['2021_acc'].mean())
    print(month_df['2019_f1'].mean(), month_df['2020_f1'].mean(), month_df['2021_f1'].mean())

    x_axis = np.arange(1,13)
    plt.figure(figsize=(10,5))
    bars = plt.bar(x_axis - 0.3, month_df['2019_acc'], 0.3, label='2019')
    plt.bar_label(bars)
    bars = plt.bar(x_axis, month_df['2020_acc'], 0.3, label='2020')
    plt.bar_label(bars)
    bars = plt.bar(x_axis + 0.3, month_df['2021_acc'], 0.3, label='2021')
    plt.bar_label(bars)
    plt.xticks(x_axis)
    plt.yticks(range(0,101,10))
    plt.xlabel('Month')
    plt.ylabel("Accuracy (%)")
    plt.title("Variation of Accuracy of XGBoost Model Across Months for Different Years")
    plt.legend()
    plt.show()

    x_axis = np.arange(1,13)
    plt.figure(figsize=(10,5))
    bars = plt.bar(x_axis - 0.3, month_df['2019_f1'], 0.3, label='2019')
    plt.bar_label(bars)
    bars = plt.bar(x_axis, month_df['2020_f1'], 0.3, label='2020')
    plt.bar_label(bars)
    bars = plt.bar(x_axis + 0.3, month_df['2021_f1'], 0.3, label='2021')
    plt.bar_label(bars)
    plt.xticks(x_axis)
    plt.yticks(range(0,101,10))
    plt.xlabel('Month')
    plt.ylabel("F1-Score (%)")
    plt.title("Variation of F1-Score of XGBoost Model Across Months for Different Years")
    plt.legend()
    plt.show()

    x_axis = np.arange(1,13)
    plt.figure(figsize=(10,5))
    bars = plt.bar(x_axis - 0.3, month_df['2019_samples'], 0.3, label='2019')
    # plt.bar_label(bars)
    bars = plt.bar(x_axis, month_df['2020_samples'], 0.3, label='2020')
    # plt.bar_label(bars)
    bars = plt.bar(x_axis + 0.3, month_df['2021_samples'], 0.3, label='2021')
    # plt.bar_label(bars)
    plt.xticks(x_axis)
    # plt.yticks(range(0,101,10))
    plt.xlabel('Month')
    plt.ylabel("Number of Sample Points")
    plt.title("Number of Samples Across Months for Different Years")
    plt.legend()
    plt.show()


def plot_month_ccd_distribution(clean_df, xgb_model, features):
    month_class = []
    df_ = deepcopy(clean_df)

    for month in range(1,13):
        df_['month'] = month
        df_ = encode(df_, 'month', 12)
        y_pred = xgb_model.predict(df_[features])
        month_class.append({
            'month': month,
            'high_class': sum(y_pred),
            'low_class': len(df_) - sum(y_pred)
        })

    month_class_df = pd.DataFrame(month_class)

    plt.figure(figsize=(8,5))
    plt.bar(month_class_df['month'], month_class_df['low_class'], color='orange', label='Low Class')
    plt.bar(month_class_df['month'], month_class_df['high_class'], bottom=month_class_df['low_class'], color='green', label='High Class')
    plt.xticks(month_class_df['month'])
    plt.xlabel("Month")
    plt.ylabel("Number of Samples")
    plt.title("Predicted Canopy Density Classes Across Months (XGBoost Model)")
    plt.legend()
    plt.show()

In [15]:
def train_acz_ccd_model(agro_zone, df, df19, df21, MODEL_PATH, threshold_percentile=0.50, threshold_val=None):
    band = 'cover'
    df[band] = df[band] * 100
    df19[band] = df19[band]*100
    df21[band] = df21[band] * 100

    clean_df = clean_data(df, band)
    clean_df19 = clean_data(df19, band)
    clean_df['year'] = 2020
    clean_df19['year'] = 2019
    clean_df = pd.concat([clean_df, clean_df19], ignore_index=True)
    print(clean_df[band].describe())
    clean_df21 = clean_data(df21, band)

    print("Median CCD Threshold: ", round(clean_df[band].median()))
    y_col = 'class_code'
    exclude_months = []

    if threshold_val is None:
      threshold = round(clean_df[band].quantile(threshold_percentile))
    else:
      threshold = round(threshold_val)

    clean_df[y_col] = clean_df[band].apply(lambda x: get_class(x, threshold))

    sample_df = clean_df[~clean_df['month'].isin(exclude_months)]
    if len(sample_df) > 200_000:
      sample_df = sample_df.sample(n=200_000, random_state=42, ignore_index=True)
    sample_df = balance_class(sample_df, y_col)
    print(sample_df[y_col].value_counts())

    X_train, X_test, y_train, y_test = train_test_split(sample_df[features], sample_df[y_col], test_size=0.2, random_state=42, shuffle=True)
    train_df = deepcopy(X_train)
    train_df[y_col] = y_train
    test_df = deepcopy(X_test)
    test_df[y_col] = y_test

    # All S1&S2 Features + month for CCD models
    print("Total Features:", len(features))
    xgb_model, pred_y_train, pred_y_test = xgb_cv(train_df, test_df, features, y_col)
    joblib.dump(xgb_model, MODEL_PATH + agro_zone.replace(" &", "").replace(" ", "_") + "_toa_monthly_" + band + "_" + str(threshold) + ".joblib")

    clean_df21[y_col] = clean_df21[band].apply(lambda x: get_class(x, threshold))
    clean_df21['pred_cover'] = xgb_model.predict(clean_df21[features])
    sample21 = clean_df21[~clean_df21['month'].isin(exclude_months)]
    print("Accuracy on 2021 data:", accuracy_score(sample21[y_col], sample21['pred_cover'])*100)
    print("F1-Score on 2021 data:", f1_score(sample21[y_col], sample21['pred_cover'], average='macro')*100)
    print(classification_report(sample21[y_col], sample21['pred_cover']))

    train_f1 = round(f1_score(train_df[y_col], pred_y_train, average='macro')*100, 1)
    test_f1 = round(f1_score(test_df[y_col], pred_y_test, average='macro')*100, 1)
    test_21 = round(f1_score(sample21[y_col], sample21['pred_cover'], average='macro')*100, 1)

    plot_month_performance(clean_df, clean_df21, xgb_model, features, y_col)
    plot_month_ccd_distribution(clean_df21, xgb_model, features)

    imp = pd.DataFrame(features, columns=['feature'])
    imp['score'] = xgb_model.feature_importances_
    imp.sort_values('score', ascending=True, ignore_index=True, inplace=True)
    imp = imp[~imp['feature'].str.contains('month')]


    imp.to_csv(MODEL_PATH + agro_zone.replace(" &", "").replace(" ", "_") + "_toa_monthly_" + band + "_" + str(threshold) + "_features.csv", index=False)
    imp = pd.read_csv(MODEL_PATH + agro_zone.replace(" &", "").replace(" ", "_") + "_toa_monthly_" + band + "_" + str(threshold) + "_features.csv")


    # Selected Features for best CCD models
    final_features = features
    for thresh in imp['score'].iloc[1:]:
        print("************************ Threshold =", thresh, " ****************************")
        # select features using threshold
        selected_features = list(imp[imp['score'] >= thresh]['feature']) + ['month_sin', 'month_cos']
        print("Total Features:", len(selected_features))

        xgb_model, pred_y_train, pred_y_test = xgb_cv(train_df, test_df, selected_features, y_col)

        clean_df21[y_col] = clean_df21[band].apply(lambda x: get_class(x, threshold))
        clean_df21['pred_cover'] = xgb_model.predict(clean_df21[selected_features])
        sample21 = clean_df21[~clean_df21['month'].isin(exclude_months)]
        print("Accuracy on 2021 data:", accuracy_score(sample21[y_col], sample21['pred_cover'])*100)
        print("F1-Score on 2021 data:", f1_score(sample21[y_col], sample21['pred_cover'], average='macro')*100)
        print(classification_report(sample21[y_col], sample21['pred_cover']))

        f1_train_curr = round(f1_score(train_df[y_col], pred_y_train, average='macro')*100, 1)
        f1_test_curr = round(f1_score(test_df[y_col], pred_y_test, average='macro')*100, 1)
        f1_21_curr = round(f1_score(sample21[y_col], sample21['pred_cover'], average='macro')*100, 1)

        if test_21 <= f1_21_curr:
            train_f1 = f1_train_curr
            test_f1 = f1_test_curr
            test_21 = f1_21_curr
            final_features = selected_features
            joblib.dump(xgb_model, MODEL_PATH + agro_zone.replace(" &", "").replace(" ", "_") + "_toa_monthly_" + band + "_" + str(threshold) + ".joblib")
            plot_month_ccd_distribution(clean_df21, xgb_model, final_features)

        print("Best model with features: ", len(final_features), train_f1, test_f1, test_21)

    # dump this in a csv for reference, also dump length of train set, test set, 2021 set
    return {
        'acz': agro_zone,
        'threshold': threshold,
        'month_features': len(final_features),
        'month_train_f1': train_f1,
        'month_test_f1': test_f1,
        'month_oos_f1': test_21,
        'train_set_len': len(train_df),
        'test_set_len': len(test_df),
        'oos_set_len': len(sample21)
    }

In [16]:
# This will currently not work since I've changed the naming convention of the joblib and csv files
def plot_acz_month_performance(agro_zone, df, df19, df21, MODEL_PATH):
    band = 'cover'
    df[band] = df[band] * 100
    df19[band] = df19[band]*100
    df21[band] = df21[band] * 100

    clean_df = clean_data(df, band)
    clean_df19 = clean_data(df19, band)
    clean_df['year'] = 2020
    clean_df19['year'] = 2019
    clean_df = pd.concat([clean_df, clean_df19], ignore_index=True)
    print(clean_df[band].describe())
    clean_df21 = clean_data(df21, band)

    print("Median CCD Threshold: ", round(clean_df[band].median()))

    y_col = 'class_code'
    threshold = round(clean_df[band].median())
    exclude_months = []

    clean_df[y_col] = clean_df[band].apply(lambda x: get_class(x, threshold))

    sample_df = clean_df[~clean_df['month'].isin(exclude_months)]
    if len(sample_df) > 200_000:
      sample_df = sample_df.sample(n=200_000, random_state=42, ignore_index=True)
    sample_df = balance_class(sample_df, y_col)
    print(sample_df[y_col].value_counts())

    X_train, X_test, y_train, y_test = train_test_split(sample_df[features], sample_df[y_col], test_size=0.2, random_state=42, shuffle=True)
    train_df = deepcopy(X_train)
    train_df[y_col] = y_train
    test_df = deepcopy(X_test)
    test_df[y_col] = y_test

    # All S1&S2 Features + month for CCD models
    print("Total Features:", len(features))
    xgb_model = joblib.load(MODEL_PATH + agro_zone.replace(" &", "").replace(" ", "_") + "_toa_monthly_" + band + "_" + str(threshold) +".joblib")
    selected_features = list(xgb_model.feature_names_in_)
    print("Total Selected Features:", len(selected_features))

    clean_df21[y_col] = clean_df21[band].apply(lambda x: get_class(x, threshold))
    clean_df21['pred_cover'] = xgb_model.predict(clean_df21[selected_features])
    sample21 = clean_df21[~clean_df21['month'].isin(exclude_months)]
    print("Accuracy on 2021 data:", accuracy_score(sample21[y_col], sample21['pred_cover'])*100)
    print("F1-Score on 2021 data:", f1_score(sample21[y_col], sample21['pred_cover'], average='macro')*100)
    print(classification_report(sample21[y_col], sample21['pred_cover']))

    plot_month_performance(clean_df, clean_df21, xgb_model, selected_features, y_col)
    plot_month_ccd_distribution(clean_df21, xgb_model, selected_features)

In [17]:
def ndvi_stats(agro_zone, df, df19, df21):
  # ndvi_f1 = {'acz': agro_zone}
  band = "cover"
  df[band] = df[band] * 100
  df19[band] = df19[band] * 100
  df21[band] = df21[band] * 100

  clean_df = clean_data(df, band)
  clean_df19 = clean_data(df19, band)
  clean_df['year'] = 2020
  clean_df19['year'] = 2019
  clean_df = pd.concat([clean_df, clean_df19], ignore_index=True)
  print(clean_df[band].describe())
  clean_df21 = clean_data(df21, band)

  print("Median", band, "Threshold: ", round(clean_df[band].median()))

  y_col = 'class_code'
  threshold = round(clean_df[band].median())
  clean_df[y_col] = clean_df[band].apply(lambda x: get_class(x, threshold))

  max_diff = 0
  ndvi_threshold = 0
  ndvi_band = "NDVI_zaid"
  for y in ['NDVI_kharif', 'NDVI_rabi', 'NDVI_zaid']:
    plt.boxplot([clean_df[clean_df['class_code']==0][y], clean_df[clean_df['class_code']==1][y]])
    plt.xticks([1, 2], ['Low', 'High'])
    plt.ylabel(y)
    plt.title(y.split("_")[-1])
    plt.show()

    med_0 = clean_df[clean_df['class_code']==0][y].median()
    med_1 = clean_df[clean_df['class_code']==1][y].median()
    diff = med_1 - med_0
    med = (med_0+med_1)/2
    if diff > max_diff:
      max_diff = diff
      ndvi_threshold = med
      ndvi_band = y
  print(agro_zone, band, ndvi_band, max_diff, ndvi_threshold)

  clean_df21['ndvi_class'] = clean_df21[ndvi_band].apply(lambda x: 0 if x <= ndvi_threshold else 1)
  clean_df21[y_col] = clean_df21[band].apply(lambda x: get_class(x, threshold))

  print(classification_report(clean_df21['class_code'], clean_df21['ndvi_class']))
  # ndvi_f1[band] = round(f1_score(clean_df21['class_code'], clean_df21['ndvi_class'], average='macro')*100, 1)
  return {
      'acz': agro_zone,
      band: round(f1_score(clean_df21['class_code'], clean_df21['ndvi_class'], average='macro')*100, 1)
      }

# **Retrain Models on ACZ specific districts data**

#### Define the valid MODEL_PATH where you want to save the trained models in **"get_model_paths"** function. Also, change the input data paths, if they are changed.

In [18]:
monthly_model_results = []
GEDI_DATA = 'CC'
MODEL_PATH, dist_data_path, dist_data_path_19, dist_data_path_21, gedi_features = get_model_paths(GEDI_DATA)

In [19]:
# model_files = glob(MODEL_PATH + "*cover*.joblib")
# len(model_files)

In [20]:
acz_list = [
    'Western Himalayan Region',
    'Eastern Himalayan Region',
    'Lower Gangetic Plain Region',
    'Middle Gangetic Plain Region',
    'Upper Gangetic Plain Region',
    'Trans Gangetic Plain Region',
    'Eastern Plateau & Hills Region',
    'Central Plateau & Hills Region',
    'Western Plateau and Hills Region',
    'Southern Plateau and Hills Region',
    'East Coast Plains & Hills Region'
    ]

# number between 0-1 indicates percentile, number between 1-100 indicates canopy density percentage
threshold_dict = {
    'Western Himalayan Region': 0.25,
    'Eastern Himalayan Region': 50, # 40-50% CD
    'Lower Gangetic Plain Region': 0.75,
    'Middle Gangetic Plain Region': 0.75,
    'Upper Gangetic Plain Region': 40, # 40% CD else 25th percentile
    'Trans Gangetic Plain Region': 0.75,
    'Eastern Plateau & Hills Region': 0.75,
    'Central Plateau & Hills Region': 0.75,
    'Western Plateau and Hills Region': 80, # 80% CD
    'Southern Plateau and Hills Region': 0.75,
    'East Coast Plains & Hills Region': 40 # 40% CD else 25th percentile
}

# acz_list = ['Western Himalayan Region']

In [21]:
data_dict = {}

for i, agro_zone in enumerate(acz_list):
  print("\n************************", agro_zone, "*****************************")
  files = glob(dist_data_path + "*.csv")
  df = get_rs_data(agro_zone, dist_data_path, files, gedi_features)

  files_19 = glob(dist_data_path_19 + "*.csv")
  df19 = get_rs_data(agro_zone, dist_data_path_19, files_19, gedi_features)

  files_21 = glob(dist_data_path_21 + "*.csv")
  df21 = get_rs_data(agro_zone, dist_data_path_21, files_21, gedi_features)

  data_dict[agro_zone] = {
    'df': df,
    'df19': df19,
    'df21': df21
    }



************************ Western Himalayan Region *****************************
Total Districts:  56
Total Sentinel-1 Data of  kharif :  317121
Total Sentinel-2 Data of  kharif :  138312
Total Sentinel-1 Data of  rabi :  317121
Total Sentinel-2 Data of  rabi :  142199
Total Sentinel-1 Data of  zaid :  317121
Total Sentinel-2 Data of  zaid :  317121
Merged Data Size:  137750
Total Districts:  56
Total Sentinel-1 Data of  kharif :  372490
Total Sentinel-2 Data of  kharif :  185604
Total Sentinel-1 Data of  rabi :  372490
Total Sentinel-2 Data of  rabi :  171709
Total Sentinel-1 Data of  zaid :  372490
Total Sentinel-2 Data of  zaid :  197792
Merged Data Size:  167129
Total Districts:  56
Total Sentinel-1 Data of  kharif :  329882
Total Sentinel-2 Data of  kharif :  298409
Total Sentinel-1 Data of  rabi :  329882
Total Sentinel-2 Data of  rabi :  329882
Total Sentinel-1 Data of  zaid :  329882
Total Sentinel-2 Data of  zaid :  329882
Merged Data Size:  298161

************************ Ea

In [22]:
import csv
import os

output_file = os.path.join(MODEL_PATH, "acz_best_model_stats.csv")
first_write = True  # To write header only once

for agro_zone in acz_list:
  print("\n************************", agro_zone, "*****************************")
  threshold = threshold_dict[agro_zone]
  print('Thresholding at', threshold)
  if threshold <= 1:
    best_model = train_acz_ccd_model(agro_zone, data_dict[agro_zone]['df'], data_dict[agro_zone]['df19'], data_dict[agro_zone]['df21'],
                                   MODEL_PATH, threshold_percentile=threshold)
  else:
    best_model = train_acz_ccd_model(agro_zone, data_dict[agro_zone]['df'], data_dict[agro_zone]['df19'], data_dict[agro_zone]['df21'],
                                   MODEL_PATH, threshold_val=threshold)

  write_header = first_write and not os.path.exists(output_file)
  with open(output_file, mode='a', newline='') as csvfile:
      writer = csv.DictWriter(csvfile, fieldnames=best_model.keys())
      if write_header:
          writer.writeheader()
          first_write = False
      writer.writerow(best_model)

  # plot_acz_month_performance(agro_zone, df, df19, df21, MODEL_PATH)

Output hidden; open in https://colab.research.google.com to view.